# Symbolic Calculations in High Energy Physics
### **LM-JEPA for Squared Amplitude Calculation**

**Submitted by:** ARNABI DUTTA

This notebook documents the pipeline for predicting Quantum Electrodynamics (QED) and Quantum Chromodynamics (QCD) squared amplitudes using a Joint Embedding Predictive Architecture (LM-JEPA). 

Unlike standard text-based sequence-to-sequence models, this pipeline introduces three core data and representation experiments:
1. **Notation Topology (`--notation`):** Comparing standard `infix` mathematical representation against 1D Polish `prefix` notation to evaluate which structure reduces sequence length and improves algebraic learning.
2. **Index Permutation Augmentation (`--augment_multiplier`):** Exploiting the physical permutation symmetry of Feynman diagrams to dynamically expand the data distribution and prevent data starvation.
3. **LLM-JEPA Objective:** Following the official implementation, utilizing dual forward-passes to align continuous target representations via Cosine Similarity rather than relying entirely on string memorization.


In [1]:
import os
import sys
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('./'))
from data.tokenizer import PhysicsAwareTokenizer
from models.llm_jepa import LLM_JEPA

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Execution Device: {device}")

Execution Device: cuda


### Preprocessing & Symmetry Augmentation

High-Energy Physics equations suffer from data starvation and arbitrary index naming. However, Feynman diagrams possess **Permutation Symmetry**

Permutation symmetry in Feynman diagrams relates to the indistinguishability of internal vertices and lines, determining the symmetry factor that acts as a divisor to the overall amplitude. These symmetries arise from rearranging vertices or propagators without changing the diagram's topology, accounting for multiple identical Wick contractions.
Reference- https://arxiv.org/abs/1011.4142



**Splitting & Augmentation :**
To prevent data leakage, the raw dataset is strictly split into an 80/10/10 ratio (Train/Val/Test) *before* any augmentation occurs. Once isolated, the training set is augmented using the `--augment_multiplier` flag. The preprocessing script (`scripts/01_preprocess_data.py`) extracts and protects physical kinematic variables (like $s_{12}, m_e, p_3$), and applies a seeded random shuffle to generic indices (e.g., `%gam_115`). This artificially expands the dataset by a factor of 15x, creating a robust training distribution keeping mathematical exactness.

In [6]:
# Execute the preprocessing and augmentation pipeline
!python scripts/01_preprocess_data.py --raw_data_dir "/media/kavinder/hdd/Arnabi/SYMBA-Test Data" --augment_multiplier 15 --output_dir ./data/processed


Parsing QCD-2-to-2-diag-TreeLevel-0.txt: 46it [00:00, 291.21it/s]
Parsing QCD-2-to-2-diag-TreeLevel-1.txt: 42it [00:00, 299.39it/s]
Parsing QCD-2-to-2-diag-TreeLevel-2.txt: 38it [00:00, 275.22it/s]
Parsing QCD-2-to-2-diag-TreeLevel-3.txt: 34it [00:00, 276.95it/s]
Parsing QCD-2-to-2-diag-TreeLevel-4.txt: 30it [00:00, 260.33it/s]
Parsing QCD-2-to-2-diag-TreeLevel-5.txt: 26it [00:00, 235.19it/s]
Parsing QCD-2-to-2-diag-TreeLevel-6.txt: 18it [00:00, 235.46it/s]
Parsing QED-2-to-2-diag-TreeLevel-0.txt: 54it [00:00, 791.67it/s]
Parsing QED-2-to-2-diag-TreeLevel-1.txt: 50it [00:00, 891.04it/s]
Parsing QED-2-to-2-diag-TreeLevel-2.txt: 46it [00:00, 781.04it/s]
Parsing QED-2-to-2-diag-TreeLevel-3.txt: 42it [00:00, 760.14it/s]
Parsing QED-2-to-2-diag-TreeLevel-4.txt: 38it [00:00, 718.28it/s]
Parsing QED-2-to-2-diag-TreeLevel-5.txt: 34it [00:00, 708.08it/s]
Parsing QED-2-to-2-diag-TreeLevel-6.txt: 30it [00:00, 711.15it/s]
Parsing QED-2-to-2-diag-TreeLevel-7.txt: 26it [00:00, 660.47it/s]
Parsing QE

### Tokenizer

Standard NLP tokenizers (like BPE or WordPiece) shatter mathematical tensors into meaningless sub-word fragments. To preserve the physics, this pipeline utilizes a custom regex lexer engineered to swallow subscripts, arguments, and superscripts whole, treating complex kinematic strings as single vocabulary tokens.

Raw equations undergo a rigorous standardization pass (e.g., mapping `**` to `^`, complex conjugations to `^CONJ`, and domain variables like `s_12` to `MANDELSTAM_12`).

**The Notation Toggle (`--notation`):**
When generating the vocabulary (`scripts/02_build_vocab.py`) and processing the data, we experiment with two modes:
* `--notation infix`: The standard human-readable mathematical format.
* `--notation prefix`: The tokens are processed through the Shunting-Yard algorithm to generate Polish Prefix notation, evaluating if 1D operator-first hierarchy improves the model's structural learning.

In [8]:
!python3 scripts/02_build_vocab.py --physics_model QCD

Loading data from /media/kavinder/hdd/Arnabi/symba-hep-lm-jepa/data/processed/QCD_train.csv...
Verified: Loaded 2808 training samplfPes!
Processing squared amplitude vocab: 100%|█| 2808/2808 [00:00<00:00, 4654.85it/s]
Shared vocab built in 1.23s, unique tokens: 2684
Tokenizer and Vocab saved to /media/kavinder/hdd/Arnabi/symba-hep-lm-jepa/data/processed for QCD
Total QCD Vocabulary Size: 2690


In [9]:
!python3 scripts/02_build_vocab.py --physics_model QED

Loading data from /media/kavinder/hdd/Arnabi/symba-hep-lm-jepa/data/processed/QED_train.csv...
Verified: Loaded 4317 training samplfPes!
Processing amplitude vocab: 100%|████████| 4317/4317 [00:00<00:00, 13973.26it/s]
Processing squared amplitude vocab: 100%|█| 4317/4317 [00:00<00:00, 19476.85it/s
Shared vocab built in 0.53s, unique tokens: 289
Tokenizer and Vocab saved to /media/kavinder/hdd/Arnabi/symba-hep-lm-jepa/data/processed for QED
Total QED Vocabulary Size: 295


### DataLoader & Masking
To implement the LM-JEPA objective, the model must learn deep, continuous representations without being exposed to the gt. 

In `data/dataset.py`, we construct a additive causal mask matrix initialized to $-\infty$. We punch out specific topological triangles to allow the Amplitude to see itself, and the Squared Amplitude to see itself, but strictly forbid cross-view attention during the JEPA forward pass.


### LLM-JEPA Architecture

The model processes the data through two simultaneous forward passes as mentioned in the paper:
1. **The Generative Pass:** Using standard causal masking, the model predicts the next token, minimizing Cross-Entropy ($L_{CE}$).
2. **The JEPA Pass:** Using the blinded topological mask, the model extracts the final hidden state of the Amplitude ($h_A$) and the Squared Amplitude ($h_{A^2}$). We apply $L_2$ normalization and compute the Cosine Similarity to align the continuous manifolds.

$$L_{total} = L_{CE} + \lambda_{JEPA} \left( 1 - \cos(h_A, h_{A^2}) \right)$$

`models/llm_jepa.py`
```python
text_rep = F.normalize(hidden_states_jepa[i, t_end_idx, :], p=2, dim=0)
code_rep = F.normalize(hidden_states_jepa[i, c_end_idx, :], p=2, dim=0)

cos_sim = F.cosine_similarity(text_rep.unsqueeze(0), code_rep.unsqueeze(0))
jepa_loss += (1.0 - cos_sim.squeeze())

The script below initiates the distributed training loop via Weights & Biases, utilizing mixed-precision (`torch.autocast`) and Cosine Annealing. 

*Note: Training on the full augmented dataset takes several hours. For the convenience, this command is commented out. We provide the pre-trained weights for instant evaluation.*

In [13]:
# OPTIONAL: To train the model from scratch, uncomment the line below.
# !python scripts/03_train.py --physics_model QCD --mode jepa

### Evaluation
Because symbolic math is commutative, standard token accuracy heavily penalizes structurally valid but reordered equations. We evaluate the models using 4 criteria:
1. **Sequence Accuracy:** Strict exact string match.
2. **Token Accuracy:** Positional token-by-token alignment.
3. **Structural Match (Levenshtein):** Measures edit distance to identify syntactically valid but shifted mathematical trees.
4. **Algebraic Match (SymPy):** Generated outputs are parsed through a SymPy solver (`sympy.simplify(pred - gt) == 0`) to explicitly verify mathematical equivalence.

**Prefix Reversal:** SymPy requires standard mathematical formatting. To evaluate models trained on the `--notation prefix` toggle, the inference script utilizes a reverse Shunting-Yard parser using a stack-based algorithm to reconstruct human-readable equations before feeding them to the SymPy solver. A strict 2-second `signal.alarm` timeout prevents the CPU from hanging on hallucinated, invalid topologies. 

Below, we download the pre-trained weights, instantiate the JEPA model, and run the evaluation script.

In [1]:
import os

# Create checkpoint directory
# os.makedirs("logs/jepa/checkpoints/QCD", exist_ok=True)
# ckpt_path = "logs/jepa/checkpoints/QCD/best_model.pt"

# Download the pre-trained weights
# if not os.path.exists(ckpt_path):
#     print("Downloading pre-trained LLM-JEPA weights")
#     # gdown.download('LINK TO BE UPLOADED', ckpt_path, quiet=False)
#     print("Weights downloaded successfully!")

# Run the lightning-fast evaluation script
!python ./inference/evaluate.py --physics_model QED --mode jepa --wt_path ./logs/jepa/checkpoints/QED/best_model_QED_jepa_prefix_mult1.pt --notation prefix

Evaluation for QED on cuda
Loading test data, tokenizer, and vocabulary
Loading model weights from: ./logs/jepa/checkpoints/QED/best_model_QED_jepa_prefix_mult1.pt
Loaded checkpoint from epoch 75 | val_loss=0.363335

Evaluating 36 samples
Inference: 100%|████████████████████████████████| 36/36 [00:02<00:00, 13.52it/s]

QED EVALUATION METRICS (JEPA)
Total Samples:       36
1. Sequence Acc:     0.00% (0/36)  
2. Token Accuracy:   35.19%
3. Algebraic Match:  88.89% (32/36) 
4. Structural (Lev): 67.22%
5. Inference Speed:  1290.20 tokens/sec

[ RANDOM SAMPLES ]

--- Sample 1 ---
AMP  (Input) : 1/9*i*e^2*gamma_{+%\sigma_980,%eta_587,%eps_277}*gamma_{%\sigma_980,%gam_499,%del_385}*s_{k_318,%eps_277}(p_1)_u*s_{l_318,%eta_587}(p_2)_v^(*)*tt_{j_320,%del_385}(p_4)_v*tt_{i_320,%gam_499}(p_3)_u^(*)/(m_s^2 + s_12 + 1/2*reg_prop)
GT   (SqAmp) : div 1 mul 81 mul E_CHARGE^4 mul add mul 16 mul m_s^2 m_tt^2 add mul 8 mul m_tt^2 MANDELSTAM_12 add mul 8 mul MANDELSTAM_14 MANDELSTAM_23 add mul 8 mul MANDEL

In [1]:
!python ./inference/evaluate.py --physics_model QED --mode jepa --wt_path ./logs/jepa/checkpoints/QED/best_model_QED_jepa_prefix_mult15.pt --augment_multiplier 15 --notation prefix

Evaluation for QED on cuda
Loading test data, tokenizer, and vocabulary
Loading model weights from: ./logs/jepa/checkpoints/QED/best_model_QED_jepa_prefix_mult15.pt
Loaded checkpoint from epoch 75 | val_loss=0.000755

Evaluating 36 samples
Inference: 100%|████████████████████████████████| 36/36 [00:01<00:00, 20.57it/s]

QED EVALUATION METRICS (JEPA)
Total Samples:       36
1. Sequence Acc:     94.44% (34/36)  
2. Token Accuracy:   98.11%
3. Algebraic Match:  100.00% (36/36)  
4. Structural (Lev): 99.82%
5. Inference Speed:  1496.53 tokens/sec

[ RANDOM SAMPLES ]

--- Sample 1 ---
AMP  (Input) : -i*e^2*(p_1_%\nu_28037*gamma_{%\mu_27603,%eta_9617,%eps_7326}*A_{l_10695,+%\mu_27603}(p_2)*A_{j_10697,+%\nu_28037}(p_4)^(*)*mu_{i_10697,%eps_7326}(p_3)_v*mu_{k_10695,%eta_9617}(p_1)_v^(*) + -1/2*p_4_%\tau_27567*gamma_{%\mu_27603,%eta_9638,%eps_7328}*gamma_{%\nu_28037,%eta_9620,%eta_9637}*gamma_{+%\tau_27567,%eta_9637,%eta_9638}*A_{l_10695,+%\mu_27603}(p_2)*A_{j_10697,+%\nu_28037}(p_4)^(*)*mu_{i_

In [2]:
!python ./inference/evaluate.py --physics_model QCD --mode jepa --wt_path ./logs/jepa/checkpoints/QCD/best_model_QCD_jepa_prefix_mult1.pt --augment_multiplier 1 --notation prefix

Evaluation for QCD on cuda
Loading test data, tokenizer, and vocabulary
Loading model weights from: ./logs/jepa/checkpoints/QCD/best_model_QCD_jepa_prefix_mult1.pt
Loaded checkpoint from epoch 108 | val_loss=0.853821

Evaluating 24 samples
Inference: 100%|████████████████████████████████| 24/24 [00:10<00:00,  2.33it/s]

QCD EVALUATION METRICS (JEPA)
Total Samples:       24
1. Sequence Acc:     0.00% (0/24)  
2. Token Accuracy:   9.45%
3. Algebraic Match:  50.00% (12/24)  
4. Structural (Lev): 39.74%
5. Inference Speed:  710.58 tokens/sec

[ RANDOM SAMPLES ]

--- Sample 1 ---
AMP  (Input) : 1/12*i*g^2*gamma_{+%\tau_9025,%eta_2410,%beta_1544}*gamma_{%\tau_9025,%gam_6639,%gam_6640}*(b_{k_2032,%F_1167,%eta_2410}(p_4)_u^(*)*b_{i_2028,%F_1167,%beta_1544}(p_2)_u*s_{j_2032,%G_3705,%gam_6640}(p_3)_v*s_{l_2026,%G_3705,%gam_6639}(p_1)_v^(*) + (-3)*b_{k_2032,%H_4067,%eta_2410}(p_4)_u^(*)*b_{i_2028,%G_3706,%beta_1544}(p_2)_u*s_{j_2032,%H_4067,%gam_6640}(p_3)_v*s_{l_2026,%G_3706,%gam_6639}(p_1)_v^(*

In [4]:
!python ./inference/evaluate.py --physics_model QCD --mode jepa --wt_path ./logs/jepa/checkpoints/QCD/best_model_QCD_jepa_prefix_mult15.pt --augment_multiplier 15 --notation prefix

Evaluation for QCD on cuda
Loading test data, tokenizer, and vocabulary
Loading model weights from: ./logs/jepa/checkpoints/QCD/best_model_QCD_jepa_prefix_mult15.pt
Loaded checkpoint from epoch 137 | val_loss=0.028463

Inference: 100%|████████████████████████████████| 24/24 [00:06<00:00,  3.77it/s]

QCD EVALUATION METRICS (JEPA)
Total Samples:       24
1. Sequence Acc:     79.17% (19/24) 
2. Token Accuracy:   87.13%
3. Algebraic Match:  79.17% (19/24) 
4. Structural (Lev): 93.58%
5. Inference Speed:  692.37 tokens/sec

[ RANDOM SAMPLES ]

--- Sample 1 ---
AMP  (Input) : 1/12*i*g^2*gamma_{+%\tau_347,%gam_325,%eps_78}*gamma_{%\tau_347,%gam_326,%del_319}*(s_{j_80,%H_103,%gam_326}(p_3)_u^(*)*s_{k_80,%H_103,%del_319}(p_4)_v*u_{i_80,%C_102,%gam_325}(p_2)_v^(*)*u_{l_78,%C_102,%eps_78}(p_1)_u + (-3)*s_{j_80,%C_103,%gam_326}(p_3)_u^(*)*s_{k_80,%D_139,%del_319}(p_4)_v*u_{i_80,%D_139,%gam_325}(p_2)_v^(*)*u_{l_78,%C_103,%eps_78}(p_1)_u)/(m_u^2 + s_12 + 1/2*reg_prop)
GT   (SqAmp) : add div neg 1 mu

In [6]:
!python ./inference/evaluate.py --physics_model QED --mode jepa --wt_path ./logs/jepa/checkpoints/QED/best_model_QED_jepa_infix_mult1.pt --augment_multiplier 1 --notation infix

Evaluation for QED on cuda
Loading test data, tokenizer, and vocabulary
Loading model weights from: ./logs/jepa/checkpoints/QED/best_model_QED_jepa_infix_mult1.pt
Loaded checkpoint from epoch 211 | val_loss=0.069056

Evaluating 36 samples
Inference: 100%|████████████████████████████████| 36/36 [00:02<00:00, 13.01it/s]

QED EVALUATION METRICS (JEPA)
Total Samples:       36
1. Sequence Acc:     63.89% (23/36)  
2. Token Accuracy:   81.94%
3. Algebraic Match:  63.89% (23/36)  
4. Structural (Lev): 96.51%
5. Inference Speed:  1151.13 tokens/sec

[ RANDOM SAMPLES ]

--- Sample 1 ---
AMP  (Input) : 4/9*i*e^2*(p_3_%\nu_21863*gamma_{%\mu_22633,%eta_9132,%eta_9133}*A_{l_8391,+%\mu_22633}(p_2)*A_{j_8393,+%\nu_21863}(p_4)^(*)*u_{i_8393,%eta_9133}(p_3)_v*u_{k_8391,%eta_9132}(p_1)_v^(*) + 1/2*p_4_%\tau_21803*gamma_{%\mu_22633,%eta_9137,%eta_9173}*gamma_{%\nu_21863,%eta_9174,%eta_9138}*gamma_{+%\tau_21803,%eta_9173,%eta_9174}*A_{l_8391,+%\mu_22633}(p_2)*A_{j_8393,+%\nu_21863}(p_4)^(*)*u_{i_8393,%eta

In [7]:
!python ./inference/evaluate.py --physics_model QED --mode jepa --wt_path ./logs/jepa/checkpoints/QED/best_model_QED_jepa_infix_mult15.pt --augment_multiplier 15 --notation infix

Evaluation for QED on cuda
Loading test data, tokenizer, and vocabulary
Loading model weights from: ./logs/jepa/checkpoints/QED/best_model_QED_jepa_infix_mult15.pt
Loaded checkpoint from epoch 53 | val_loss=0.001436

Evaluating 36 samples
Inference: 100%|████████████████████████████████| 36/36 [00:02<00:00, 14.81it/s]

QED EVALUATION METRICS (JEPA)
Total Samples:       36
1. Sequence Acc:     94.44% (34/36)  <-- Exact Match
2. Token Accuracy:   99.69%
3. Algebraic Match:  94.44% (34/36)  <-- SymPy Verified
4. Structural (Lev): 99.69%
5. Inference Speed:  1267.41 tokens/sec

[ RANDOM SAMPLES ]

--- Sample 1 ---
AMP  (Input) : -4/9*i*e^2*(p_3_%\rho_8366*gamma_{%\nu_8471,%eta_2937,%alpha_2243}*A_{l_3123,+%\nu_8471}(p_2)*A_{j_3125,+%\rho_8366}(p_4)^(*)*c_{i_3125,%eta_2937}(p_3)_u^(*)*c_{k_3123,%alpha_2243}(p_1)_u + 1/2*p_4_%\nu_8473*gamma_{%\nu_8471,%eta_2958,%alpha_2245}*gamma_{+%\nu_8473,%eta_2957,%eta_2958}*gamma_{%\rho_8366,%eta_2940,%eta_2957}*A_{l_3123,+%\nu_8471}(p_2)*A_{j_3125,+%\r

In [8]:
!python ./inference/evaluate.py --physics_model QCD --mode jepa --wt_path ./logs/jepa/checkpoints/QCD/best_model_QCD_jepa_infix_mult1.pt --augment_multiplier 1 --notation infix

Evaluation for QCD on cuda
Loading test data, tokenizer, and vocabulary
Loading model weights from: ./logs/jepa/checkpoints/QCD/best_model_QCD_jepa_infix_mult1.pt
Loaded checkpoint from epoch 141 | val_loss=0.927944

Evaluating 24 samples
Inference: 100%|████████████████████████████████| 24/24 [00:10<00:00,  2.39it/s]

QCD EVALUATION METRICS (JEPA)
Total Samples:       24
1. Sequence Acc:     0.00% (0/24) 
2. Token Accuracy:   11.20%
3. Algebraic Match:  0.00% (0/24) 
4. Structural (Lev): 32.29%
5. Inference Speed:  686.49 tokens/sec

[ RANDOM SAMPLES ]

--- Sample 1 ---
AMP  (Input) : -i*g^2*(p_2_%\rho_923*gamma_{%\nu_810,%eta_1003,%eps_446}*T_C_10_{%b_134,%D_450,%D_4866}*T_C_10_{%h_235,%D_4866,%E_526}*G_{j_196,%h_235,+%\nu_810}(p_3)^(*)*G_{k_196,%b_134,+%\rho_923}(p_4)^(*)*d_{l_194,%E_526,%eps_446}(p_1)_u*d_{i_196,%D_450,%eta_1003}(p_2)_v^(*) + -1/2*p_4_%\rho_925*gamma_{%\nu_810,%eta_1034,%eps_448}*gamma_{%\rho_923,%eta_1006,%eta_1033}*gamma_{+%\rho_925,%eta_1033,%eta_1034}*T_C_10_{%

In [1]:
!python ./inference/evaluate.py --physics_model QCD --mode jepa --wt_path ./logs/jepa/checkpoints/QCD/best_model_QCD_jepa_infix_mult15.pt --augment_multiplier 15 --notation infix

Evaluation for QCD on cuda
Loading test data, tokenizer, and vocabulary
Loading model weights from: ./logs/jepa/checkpoints/QCD/best_model_QCD_jepa_infix_mult15.pt
Loaded checkpoint from epoch 134 | val_loss=0.036520

Evaluating 24 samples
Inference: 100%|████████████████████████████████| 24/24 [00:05<00:00,  4.22it/s]

QCD infix EVALUATION METRICS (JEPA) (15) 
Total Samples:       24
1. Sequence Acc:     20.83% (5/24)  <-- Exact Match
2. Token Accuracy:   64.74%
3. Algebraic Match:  20.83% (5/24)  <-- SymPy Verified
4. Structural (Lev): 88.21%
5. Inference Speed:  878.63 tokens/sec

[ RANDOM SAMPLES ]

--- Sample 1 ---
AMP  (Input) : -i*g^2*(p_3_%\tau_3122*gamma_{%\rho_9166,%eta_7231,%eps_6099}*T_C_10_{%f_752,%H_4984,%D_7765}*T_C_10_{%f_753,%D_7765,%E_539}*G_{k_1028,%f_752,+%\tau_3122}(p_4)^(*)*G_{i_1024,%f_753,+%\rho_9166}(p_2)*b_{j_1024,%H_4984,%eta_7231}(p_3)_u^(*)*b_{l_1018,%E_539,%eps_6099}(p_1)_u + 1/2*p_4_%\tau_3124*gamma_{%\rho_9166,%eta_7262,%eps_6101}*gamma_{%\tau_3122,%eta_